<a href="https://colab.research.google.com/github/zeyniaa/autoscript-ai-assessment/blob/main/audio_analysis_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install ffmpeg dan library pendukung
!apt-get update
!apt-get install -y ffmpeg
!pip install google-genai pydantic

import subprocess
import json
import os
import re
from pydantic import BaseModel
from typing import List

# Import pengelola rahasia dari Colab
from google.colab import userdata

# Buat folder untuk menampung file audio uji coba
os.makedirs("sample_audio", exist_ok=True)
print("Setup selesai! Sistem siap digunakan.")

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:14 h

In [2]:
class AudioAnalyzer:
    def __init__(self, file_path: str):
        self.file_path = file_path

    def get_metadata(self) -> dict:
        """Mengambil durasi, bitrate, sample rate, dan channel."""
        cmd = [
            'ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_format',
            '-show_streams', self.file_path
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        data = json.loads(result.stdout)

        audio_stream = next((s for s in data.get('streams', []) if s.get('codec_type') == 'audio'), None)
        if not audio_stream:
            raise ValueError("File ini tidak memiliki stream audio yang valid.")

        return {
            "duration": float(data['format'].get('duration', 0)),
            "bitrate": int(data['format'].get('bit_rate', 0)),
            "sample_rate": int(audio_stream.get('sample_rate', 0)),
            "channels": int(audio_stream.get('channels', 0))
        }

    def analyze_quality(self) -> dict:
        """Mendeteksi durasi hening dan rata-rata volume."""
        cmd = [
            'ffmpeg', '-i', self.file_path,
            '-af', 'silencedetect=noise=-30dB:d=2,volumedetect',
            '-f', 'null', '-'
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        output = result.stderr

        # Cari pola teks hasil analisis volume
        mean_volume_match = re.search(r'mean_volume:\s*([\-\d\.]+)\s*dB', output)
        max_volume_match = re.search(r'max_volume:\s*([\-\d\.]+)\s*dB', output)

        mean_volume = float(mean_volume_match.group(1)) if mean_volume_match else 0.0
        max_volume = float(max_volume_match.group(1)) if max_volume_match else 0.0

        # Asumsi clipping terjadi jika max volume mendekati atau mencapai 0 dB
        clipping_detected = max_volume >= 0.0

        # Cari semua durasi hening dan jumlahkan
        silence_durations = re.findall(r'silence_duration:\s*([\d\.]+)', output)
        total_silence = sum(float(dur) for dur in silence_durations)

        return {
            "total_silence_seconds": total_silence,
            "clipping_detected": clipping_detected,
            "avg_volume_db": mean_volume,
            "max_volume_db": max_volume
        }

In [5]:
from google import genai
import time # Import library time untuk fitur jeda/tunggu

class AudioQualityMetrics(BaseModel):
    silence_ratio: float
    clipping_detected: bool
    avg_volume_db: float

class AssessmentReport(BaseModel):
    file_name: str
    duration_seconds: float
    audio_quality: AudioQualityMetrics
    issues: List[str]

def generate_insights(metadata: dict, quality: dict, api_key: str, max_retries: int = 3) -> List[str]:
    """Menyuruh Gemini merangkum masalah dengan mekanisme retry jika server sibuk."""
    client = genai.Client(api_key=api_key)

    prompt = f"""
    Anda adalah AI Engineer. Analisis data teknis rekaman deposisi pengadilan ini:
    - Durasi: {metadata['duration']} detik
    - Total Hening: {quality['total_silence_seconds']} detik
    - Rata-rata Volume: {quality['avg_volume_db']} dB
    - Ada Clipping?: {quality['clipping_detected']}

    Tulis maksimal 2 kalimat ringkas mengenai kualitas audio ini dan berikan rekomendasi teknis perbaikannya.
    """

    # Mekanisme Coba Lagi (Retry)
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
            )
            return [response.text.strip()]

        except Exception as e:
            print(f"  [Peringatan] Gagal memanggil LLM (Percobaan {attempt + 1}/{max_retries}). Error: {e}")
            if attempt < max_retries - 1:
                waktu_tunggu = 5 # Tunggu 5 detik sebelum coba lagi
                print(f"  [Info] Menunggu {waktu_tunggu} detik sebelum mencoba lagi...")
                time.sleep(waktu_tunggu)
            else:
                print("  [Error] LLM tetap gagal diakses setelah beberapa kali percobaan.")
                return ["Gagal menghasilkan wawasan otomatis. Server LLM saat ini sedang sibuk atau tidak tersedia."]

In [7]:
def process_single_audio(file_path: str, api_key: str) -> str:
    print(f"Memulai analisis untuk: {file_path}...\n")

    analyzer = AudioAnalyzer(file_path)
    metadata = analyzer.get_metadata()
    quality = analyzer.analyze_quality()

    # Hitung rasio keheningan (silence / total durasi)
    silence_ratio = 0.0
    if metadata['duration'] > 0:
        silence_ratio = round(quality['total_silence_seconds'] / metadata['duration'], 2)

    # Minta AI memberikan insight
    issues = generate_insights(metadata, quality, api_key)

    # Bungkus menjadi JSON terstruktur
    final_report = AssessmentReport(
        file_name=os.path.basename(file_path),
        duration_seconds=round(metadata['duration'], 2),
        audio_quality=AudioQualityMetrics(
            silence_ratio=silence_ratio,
            clipping_detected=quality['clipping_detected'],
            avg_volume_db=quality['avg_volume_db']
        ),
        issues=issues
    )

    return final_report.model_dump_json(indent=2)

# ==========================================
# BAGIAN PENGUJIAN (TESTING)
# ==========================================
# 1. Ambil API Key dari Secrets Colab
GEMINI_KEY = userdata.get('GEMINI_API_KEY')

# 2. Upload file ".mp3" dari Google Drive soal ke folder "sample_audio" di Colab
TARGET_FILE = "sample_audio/moonlight-plaza.mp3"

# 4. Eksekusi program jika file sudah ada
if os.path.exists(TARGET_FILE):
    hasil_json = process_single_audio(TARGET_FILE, GEMINI_KEY)
    print("=== HASIL ANALISIS ===")
    print(hasil_json)
else:
    print(f"File {TARGET_FILE} belum ditemukan. Jangan lupa upload file audionya ya!")

Memulai analisis untuk: sample_audio/moonlight-plaza.mp3...

=== HASIL ANALISIS ===
{
  "file_name": "moonlight-plaza.mp3",
  "duration_seconds": 854.57,
  "audio_quality": {
    "silence_ratio": 0.01,
    "clipping_detected": false,
    "avg_volume_db": -25.3
  },
  "issues": [
    "Kualitas audio ini menunjukkan volume rata-rata yang rendah dan hening yang sangat minim, mengindikasikan rasio signal-to-noise (SNR) yang buruk yang dapat menghambat akurasi AI, meskipun tidak ada clipping.\n\nRekomendasi teknis adalah menerapkan proses post-processing meliputi peningkatan gain, pengurangan noise adaptif, dan normalisasi, untuk meningkatkan rasio signal-to-noise (SNR) demi akurasi AI yang lebih baik."
  ]
}
